In [2]:
import pandas as pd

file_path = "../data/raw/online_retail.xlsx"

df = pd.read_excel(file_path)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.shape

(541909, 8)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [5]:
df["InvoiceDate"].min(), df["InvoiceDate"].max()

(Timestamp('2010-12-01 08:26:00'), Timestamp('2011-12-09 12:50:00'))

In [6]:
df["StockCode"].nunique()

4070

In [7]:
(df["Quantity"] < 0).sum()

np.int64(10624)

In [8]:
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

In [9]:
df.shape

(530104, 8)

In [11]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [12]:
df["InvoiceDate"].min(), df["InvoiceDate"].max()

(Timestamp('2010-12-01 08:26:00'), Timestamp('2011-12-09 12:50:00'))

In [13]:
(df["Quantity"] < 0).sum()

np.int64(0)

In [14]:
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

In [15]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [16]:
sales = df[["InvoiceDate", "StockCode", "Quantity", "UnitPrice"]].copy()

sales.columns = [
    "date",
    "product_id",
    "units_sold",
    "price"
]

sales.head()

,date,product_id,units_sold,price
0,2010-12-01 08:26:00,85123A,6,2.55
1,2010-12-01 08:26:00,71053,6,3.39
2,2010-12-01 08:26:00,84406B,8,2.75
3,2010-12-01 08:26:00,84029G,6,3.39
4,2010-12-01 08:26:00,84029E,6,3.39


In [17]:
sales_daily = (
    sales
    .groupby(["date", "product_id"])
    .agg({
        "units_sold": "sum",
        "price": "mean"
    })
    .reset_index()
)

sales_daily.head()

,date,product_id,units_sold,price
0,2010-12-01 08:26:00,21730,6,4.25
1,2010-12-01 08:26:00,22752,2,7.65
2,2010-12-01 08:26:00,71053,6,3.39
3,2010-12-01 08:26:00,84029E,6,3.39
4,2010-12-01 08:26:00,84029G,6,3.39


In [18]:
sales_daily.shape

(517329, 4)

In [19]:
import numpy as np

warehouses = [f"W{i}" for i in range(1, 11)]

sales_daily["warehouse_id"] = np.random.choice(
    warehouses,
    size=len(sales_daily)
)

sales_daily.head()

,date,product_id,units_sold,price,warehouse_id
0,2010-12-01 08:26:00,21730,6,4.25,W8
1,2010-12-01 08:26:00,22752,2,7.65,W3
2,2010-12-01 08:26:00,71053,6,3.39,W5
3,2010-12-01 08:26:00,84029E,6,3.39,W6
4,2010-12-01 08:26:00,84029G,6,3.39,W10


In [20]:
output_path = "../data/processed/sales_cleaned.parquet"

sales_daily.to_parquet(output_path, index=False)

ArrowInvalid: ("Could not convert '84029E' with type str: tried to convert to int64", 'Conversion failed for column product_id with type object')

In [21]:
sales_daily["product_id"] = sales_daily["product_id"].astype(str)

In [22]:
output_path = "../data/processed/sales_cleaned.parquet"

sales_daily.to_parquet(output_path, index=False)

In [23]:
import pandas as pd

df = pd.read_parquet("data/processed/sales_cleaned.parquet")

print(df.columns)

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/sales_cleaned.parquet'

In [24]:
import os
os.getcwd()


'c:\\Users\\User\\Desktop\\PROYECTOS\\ai-logistics-copilot\\notebooks'

In [25]:
import pandas as pd

df = pd.read_parquet("../data/processed/sales_cleaned.parquet")

print(df.columns)

Index(['date', 'product_id', 'units_sold', 'price', 'warehouse_id'], dtype='str')
